In [2]:
!pip -q install pandas scikit-learn nltk matplotlib

In [3]:
import pandas as pd

news_data = {
    "title": [
        "Heavy rain disrupts city traffic",
        "Flooding closes several city roads",
        "Technology company launches new AI tool",
        "New artificial intelligence tool released",
        "Cricket team wins championship final"
    ],
    "text": [
        "Heavy rainfall caused traffic jams across the city on Monday.",
        "Flood water forced authorities to close several roads after heavy rain.",
        "A technology company launched a new artificial intelligence tool today.",
        "The company released an AI tool designed to help users work faster.",
        "The cricket team won the championship final after a close match."
    ]
}

news_df = pd.DataFrame(news_data)

print("NewsLens sample dataset created!")
print("Number of articles:", len(news_df))
display(news_df)

NewsLens sample dataset created!
Number of articles: 5


,title,text
0,Heavy rain disrupts city traffic,Heavy rainfall caused traffic jams across the ...
1,Flooding closes several city roads,Flood water forced authorities to close severa...
2,Technology company launches new AI tool,A technology company launched a new artificial...
3,New artificial intelligence tool released,The company released an AI tool designed to he...
4,Cricket team wins championship final,The cricket team won the championship final af...


In [4]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

news_df["clean_text"] = news_df["text"].apply(clean_text)

display(news_df[["title", "text", "clean_text"]])
print("Text cleaning completed successfully!")

,title,text,clean_text
0,Heavy rain disrupts city traffic,Heavy rainfall caused traffic jams across the ...,heavy rainfall caused traffic jams across the ...
1,Flooding closes several city roads,Flood water forced authorities to close severa...,flood water forced authorities to close severa...
2,Technology company launches new AI tool,A technology company launched a new artificial...,a technology company launched a new artificial...
3,New artificial intelligence tool released,The company released an AI tool designed to he...,the company released an ai tool designed to he...
4,Cricket team wins championship final,The cricket team won the championship final af...,the cricket team won the championship final af...


Text cleaning completed successfully!


In [5]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import wordpunct_tokenize

nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    tokens = wordpunct_tokenize(text)
    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    return tokens

news_df["tokens"] = news_df["clean_text"].apply(preprocess_text)

display(news_df[["title", "clean_text", "tokens"]])

print("Tokenization and stopword removal completed!")

,title,clean_text,tokens
0,Heavy rain disrupts city traffic,heavy rainfall caused traffic jams across the ...,"[heavy, rainfall, caused, traffic, jams, acros..."
1,Flooding closes several city roads,flood water forced authorities to close severa...,"[flood, water, forced, authorities, close, sev..."
2,Technology company launches new AI tool,a technology company launched a new artificial...,"[technology, company, launched, new, artificia..."
3,New artificial intelligence tool released,the company released an ai tool designed to he...,"[company, released, ai, tool, designed, help, ..."
4,Cricket team wins championship final,the cricket team won the championship final af...,"[cricket, team, championship, final, close, ma..."


Tokenization and stopword removal completed!


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

# Convert cleaned articles into numerical features
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=1000
)

tfidf_matrix = vectorizer.fit_transform(news_df["clean_text"])

# Discover 3 topics in our sample articles
nmf_model = NMF(
    n_components=3,
    random_state=42,
    max_iter=500
)

topic_matrix = nmf_model.fit_transform(tfidf_matrix)
feature_names = vectorizer.get_feature_names_out()

# Display the most important words in each topic
for topic_idx, topic in enumerate(nmf_model.components_):
    top_indices = topic.argsort()[-6:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

print("\nTopic modelling completed!")

Topic 1: tool, company, technology, launched, intelligence, artificial
Topic 2: heavy, traffic, monday, city, caused, rainfall
Topic 3: won, match, team, championship, cricket, final

Topic modelling completed!


In [7]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Measure similarity between every pair of articles
similarity_matrix = cosine_similarity(tfidf_matrix)

# Show the similarity scores in a table
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=news_df["title"],
    columns=news_df["title"]
)

print("Article similarity matrix:")
display(similarity_df.round(2))

# Find the most similar other article for each article
for i, title in enumerate(news_df["title"]):
    scores = similarity_matrix[i].copy()
    scores[i] = -1  # Exclude the article itself
    best_match = scores.argmax()

    print(f"\nArticle: {title}")
    print(f"Most similar: {news_df.iloc[best_match]['title']}")
    print(f"Similarity score: {scores[best_match]:.2f}")

print("\nSimilarity analysis completed!")

Article similarity matrix:


title,Heavy rain disrupts city traffic,Flooding closes several city roads,Technology company launches new AI tool,New artificial intelligence tool released,Cricket team wins championship final
title,,,,,
Heavy rain disrupts city traffic,1.00,0.09,0.00,0.00,0.00
Flooding closes several city roads,0.09,1.00,0.00,0.00,0.09
Technology company launches new AI tool,0.00,0.00,1.00,0.17,0.00
New artificial intelligence tool released,0.00,0.00,0.17,1.00,0.00
Cricket team wins championship final,0.00,0.09,0.00,0.00,1.00



Article: Heavy rain disrupts city traffic
Most similar: Flooding closes several city roads
Similarity score: 0.09

Article: Flooding closes several city roads
Most similar: Heavy rain disrupts city traffic
Similarity score: 0.09

Article: Technology company launches new AI tool
Most similar: New artificial intelligence tool released
Similarity score: 0.17

Article: New artificial intelligence tool released
Most similar: Technology company launches new AI tool
Similarity score: 0.17

Article: Cricket team wins championship final
Most similar: Flooding closes several city roads
Similarity score: 0.09

Similarity analysis completed!


In [8]:
from sklearn.cluster import AgglomerativeClustering
import pandas as pd

# Convert similarity scores into distances
distance_matrix = 1 - similarity_matrix
distance_matrix = distance_matrix.clip(0, 1)
distance_matrix = (distance_matrix + distance_matrix.T) / 2
distance_matrix[range(len(distance_matrix)), range(len(distance_matrix))] = 0

# Group articles using a starting distance threshold
clustering = AgglomerativeClustering(
    n_clusters=None,
    metric="precomputed",
    linkage="average",
    distance_threshold=0.85
)

news_df["event_cluster"] = clustering.fit_predict(distance_matrix)

print("News articles grouped into candidate events:")
for cluster_id, group in news_df.groupby("event_cluster"):
    print(f"\nEvent group {cluster_id + 1}")
    for title in group["title"]:
        print(" -", title)

print("\nEvent clustering completed!")

News articles grouped into candidate events:

Event group 1
 - Technology company launches new AI tool
 - New artificial intelligence tool released

Event group 2
 - Flooding closes several city roads

Event group 3
 - Cricket team wins championship final

Event group 4
 - Heavy rain disrupts city traffic

Event clustering completed!


In [9]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def extractive_summary(text, num_sentences=2):
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    sentences = [s for s in sentences if s]

    if len(sentences) <= num_sentences:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(sentences)

    scores = np.asarray(matrix.sum(axis=1)).ravel()
    top_indices = sorted(
        np.argsort(scores)[-num_sentences:]
    )

    return " ".join(sentences[i] for i in top_indices)

# Test the summarizer on the first article
sample_article = news_df.loc[0, "text"]
summary = extractive_summary(sample_article)

print("ORIGINAL ARTICLE:")
print(sample_article)

print("\nEXTRACTIVE SUMMARY:")
print(summary)

ORIGINAL ARTICLE:
Heavy rainfall caused traffic jams across the city on Monday.

EXTRACTIVE SUMMARY:
Heavy rainfall caused traffic jams across the city on Monday.


In [10]:
!pip -q install transformers sentencepiece

In [1]:
%pip install -q --upgrade transformers torch sentencepiece

In [2]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6"
)

print("Abstractive summarization model loaded successfully!")

ModuleNotFoundError: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.